<a href="https://colab.research.google.com/github/MikanQuyt/gsdm/blob/main/GSDM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/MikanQuyt/gsdm.git

Cloning into 'gsdm'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 189 (delta 47), reused 8 (delta 8), pack-reused 77 (from 1)
Receiving objects: 100% (189/189), 168.64 KiB | 1.31 MiB/s, done.
Resolving deltas: 100% (64/64), done.


In [1]:
%cd gsdm

[Errno 2] No such file or directory: 'gsdm'
/content


In [3]:
!pip install diffusers transformers accelerate opencv-python

In [4]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu121


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir -p /content/gsdm/checkpoint

In [3]:
!cp /content/drive/MyDrive/checkpoint/spm.pt /content/gsdm/checkpoint
!cp /content/drive/MyDrive/checkpoint/rm_gen.pth /content/gsdm/checkpoint

**TEST**

In [9]:
!cat /content/gsdm/config/final.yaml

name: GSDM
distributed: false
val_dataset:
  input_dir: null
  output_dir: null
  resolution: [64, 256]
gpu_ids:
- 0
RM:
  model:
    beta_schedule:  # use munual beta_schedule for acceleration
      train:
        linear_end: 0.01
        linear_start: 1.0e-06
        n_timestep: 2000
        schedule: linear
      val:
        linear_end: 0.01
        linear_start: 1.0e-06
        n_timestep: 2000
        schedule: linear
    diffusion:
      channels: 3
      conditional: true  # unconditional generation or unconditional generation
      image_size:
      - 64
      - 256
      sampling_timesteps: 1
    finetune_norm: false
    unet:
      attn_res:
      - 16
      channel_multiplier:
      - 1
      - 2
      - 4
      - 8
      - 8
      dropout: 0.2
      in_channel: 9
      inner_channel: 64
      out_channel: 3
      res_blocks: 2
    which_model_G: sr3  # use the ddpm or sr3 network structure
  path:
    log: logs
    results: results
    resume_state: null
    tb_logger: tb_

In [22]:
import yaml
import os

config_path = '/content/gsdm/config/final.yaml'

# Load the YAML file
with open(config_path, 'r') as f:
    config_data = yaml.safe_load(f)

# Force update SPM.resume_state
if 'SPM' not in config_data:
    config_data['SPM'] = {}
config_data['SPM']['resume_state'] = '/content/gsdm/checkpoint/spm.pt'

# Force update RM.path.resume_state
if 'RM' not in config_data:
    config_data['RM'] = {}
if 'path' not in config_data['RM']:
    config_data['RM']['path'] = {}
config_data['RM']['path']['resume_state'] = '/content/gsdm/checkpoint/rm_gen.pth'

# Write the modified YAML content back to the file
with open(config_path, 'w') as f:
    yaml.dump(config_data, f, default_flow_style=False, sort_keys=False)

print("final.yaml modified successfully with forced absolute paths.")

final.yaml modified successfully with forced absolute paths.


In [27]:
code_path = '/content/gsdm/inference.py'

with open(code_path, 'r') as f:
    content = f.read()

# Remove the broken inline import and just pass the 'logger' variable that was created at line 69
content = content.replace('import logging; my_model.load_network(logging.getLogger())', 'my_model.load_network(logger)')

with open(code_path, 'w') as f:
    f.write(content)

print("Scope issue fixed! Passed the existing logger to load_network.")

Scope issue fixed! Passed the existing logger to load_network.


In [28]:
# 1. Apply your syntax fix
#!sed -i "s/my_model.load_network()/my_model.load_network(None)/g" /content/gsdm/inference.py

# 2. Change into the project directory
%cd /content/gsdm

# 3. Run inference
!python inference.py --config config/final.yaml --input_dir input --output_dir output --save_sp False

/content/gsdm
2026-05-31 18:24:04,469 GSDM Initial Dataset Finished
2026-05-31 18:24:04,469 GSDM Initial Model Finished
2026-05-31 18:24:05,628 GSDM Loading pretrained model for G [./checkpoint/rm] ...
2026-05-31 18:24:06,008 GSDM Loading Model Finished
  0% 0/1 [00:00<?, ?it/s]2026-05-31 18:24:07,429 GSDM ['0000.png'] processed successfully
100% 1/1 [00:01<00:00,  1.44s/it]


**TRAIN**